# Sourcing through a middle layer, and the price of not depending on one supplier

Three mines sell ore. Two processors refine it. Two cell plants need the metal. Every leg — mine to
processor, processor to plant — has a cost per kilotonne, every mine and processor has a capacity,
and every plant has a requirement. Find the cheapest plan.

That is a transportation problem with a layer in the middle, and the middle layer brings one new
row: a processor cannot ship out what it did not take in. Then the question the module was written
for: the cheapest plan buys everything from one mine. Cap any single mine's share of the total, and
watch what that costs. The model does not say which cap is right — that is a judgement about risk —
but it turns "does diversifying cost anything?" into "how much?", which is the more useful argument.

## Setup: where the package lives

This notebook builds its models by hand and then checks them against `orteach`, the package in
`src/`. Run from a clone of the repository, `../../src` is right there. On Colab there is no clone
until this cell makes one, and no `gurobipy` until it installs it. Nothing here needs a secret.

In [1]:
import os, subprocess, sys

REPO_URL = "https://github.com/sear-labs/teaching-code"

try:
    import google.colab                      # noqa: F401 - succeeds only on Colab
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    if not os.path.isdir("/content/teaching-code"):
        subprocess.run(["git", "clone", "--quiet", REPO_URL, "/content/teaching-code"], check=True)
    os.chdir("/content/teaching-code/notebooks/13_supply_chain")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "gurobipy>=11,<14"], check=True)

sys.path.insert(0, os.path.abspath(os.path.join("..", "..", "src")))
try:
    import orteach                            # noqa: F401
except ImportError:
    raise SystemExit("orteach not found: run this notebook from its own folder inside the repository, "
                     "so that ../../src exists.")
root = os.path.abspath(os.path.join("..", ".."))
print("package:", os.path.relpath(os.path.dirname(orteach.__file__), root))

package: src\orteach


## Licence setup

Nothing here needs a key: `pip install gurobipy` ships a size-limited licence and the models below
sit well inside it. A machine with its own licence file uses that instead, and on Colab three
secrets read from the key icon in the left sidebar are used when they are there — three named here,
none contained. The environment starts silent, so no licence number lands in an output cell.

In [2]:
import gurobipy as gp
from gurobipy import GRB

env = gp.Env(empty=True)
env.setParam("OutputFlag", 0)        # start silent: the licence banner, and its licence number, stay out of the outputs
try:
    from google.colab import userdata
    try:
        env.setParam("WLSACCESSID", userdata.get("GRB_WLSACCESSID"))
        env.setParam("WLSSECRET",   userdata.get("GRB_WLSSECRET"))
        env.setParam("LICENSEID",   int(userdata.get("GRB_LICENSEID")))
        licence = "Colab Secrets (WLS)"
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        licence = "the size-limited licence pip ships"     # no key needed; see the note above
except ImportError:
    licence = "local gurobi.lic"
env.start()
print("licence:", licence)

licence: local gurobi.lic


## Four tables

Mines with capacities, processors with capacities, plants with requirements, and the routes with
their costs — ore legs and metal legs in one file with a `stage` column. Kilotonnes per year and
dollars per kilotonne throughout. Instance data, so it lives in `data/raw/` and both this notebook
and the package read it; the model looks costs up by `(origin, destination)`.

In [3]:
from orteach import tolerance
from orteach.sourcing import load_sourcing

inst = load_sourcing()

print("mines (kt/yr)      ", inst.mines)
print("processors (kt/yr) ", inst.processors)
print("plants (kt/yr)     ", inst.plants, "  total", inst.total_demand)
print(f"\n{'ore leg':26} {'$/kt':>6}      {'metal leg':30} {'$/kt':>6}")
ore, metal = list(inst.ore_cost.items()), list(inst.metal_cost.items())
for k in range(max(len(ore), len(metal))):
    left = f"{ore[k][0][0] + ' -> ' + ore[k][0][1]:26} {ore[k][1]:6.1f}" if k < len(ore) else " " * 33
    right = f"{metal[k][0][0] + ' -> ' + metal[k][0][1]:30} {metal[k][1]:6.1f}" if k < len(metal) else ""
    print(f"{left}      {right}")
print("\nore_cost[('DRC', 'China')] =", inst.ore_cost["DRC", "China"])

# to try a different tariff, edit the loaded table; the change reaches the package check at the bottom:
# inst.ore_cost["DRC", "China"] = 5.0

mines (kt/yr)       {'DRC': 120.0, 'Australia': 60.0, 'Domestic': 40.0}
processors (kt/yr)  {'China': 120.0, 'Domestic': 120.0}
plants (kt/yr)      {'Cell Plant A': 70.0, 'Cell Plant B': 50.0}   total 120.0

ore leg                      $/kt      metal leg                        $/kt
DRC -> China                  2.0      China -> Cell Plant A             3.0
DRC -> Domestic               6.0      China -> Cell Plant B             3.0
Australia -> China            4.0      Domestic -> Cell Plant A          5.0
Australia -> Domestic         5.5      Domestic -> Cell Plant B          5.0
Domestic -> China             8.0      
Domestic -> Domestic          3.0      

ore_cost[('DRC', 'China')] = 2.0


## Predict before building anything

Add up the cheapest full path from each mine to a plant. Write down which mine the cheapest plan
buys from, which processor it uses, and the total cost per kilotonne — then multiply by the total
requirement.

## Variables: one per leg

Ore on each mine-to-processor route, metal on each processor-to-plant route. Continuous,
non-negative.

In [4]:
m = gp.Model("sourcing", env=env)
tolerance.apply(m)

x = m.addVars(inst.ore_cost.keys(), lb=0.0, name="ore")       # kt, mine -> processor
y = m.addVars(inst.metal_cost.keys(), lb=0.0, name="metal")   # kt, processor -> plant
m.update()
print(m.NumVars, "variables:", len(x), "ore legs and", len(y), "metal legs")

10 variables: 6 ore legs and 4 metal legs


## Objective: every leg's cost times its flow

In [5]:
m.setObjective(x.prod(inst.ore_cost) + y.prod(inst.metal_cost), GRB.MINIMIZE)
m.update()
print(m.getObjective())

2.0 ore[DRC,China] + 6.0 ore[DRC,Domestic] + 4.0 ore[Australia,China] + 5.5 ore[Australia,Domestic] + 8.0 ore[Domestic,China] + 3.0 ore[Domestic,Domestic] + 3.0 metal[China,Cell Plant A] + 3.0 metal[China,Cell Plant B] + 5.0 metal[Domestic,Cell Plant A] + 5.0 metal[Domestic,Cell Plant B]


## The rows: capacities at both ends, requirements at the plants, and conservation in the middle

The conservation row is the one a transportation problem does not have. Say what goes wrong if it is
left out, before running the cell.

In [6]:
mine_cap = {i: m.addConstr(x.sum(i, "*") <= cap, name=f"mine[{i}]") for i, cap in inst.mines.items()}
proc_cap = {p: m.addConstr(x.sum("*", p) <= cap, name=f"processor[{p}]") for p, cap in inst.processors.items()}
balance  = {p: m.addConstr(x.sum("*", p) == y.sum(p, "*"), name=f"balance[{p}]") for p in inst.processors}
plant_req = {j: m.addConstr(y.sum("*", j) == need, name=f"plant[{j}]") for j, need in inst.plants.items()}
m.update()
for c in m.getConstrs():
    print(f"{c.ConstrName:22} {c.Sense}  {c.RHS:6.1f}")

mine[DRC]              <   120.0
mine[Australia]        <    60.0
mine[Domestic]         <    40.0
processor[China]       <   120.0
processor[Domestic]    <   120.0
balance[China]         =     0.0
balance[Domestic]      =     0.0
plant[Cell Plant A]    =    70.0
plant[Cell Plant B]    =    50.0


## Solve, and read the plan by mine and by processor

In [7]:
m.optimize()
base_cost = m.ObjVal
by_mine = {i: sum(x[i, p].X for p in inst.processors) for i in inst.mines}
by_proc = {p: sum(y[p, j].X for j in inst.plants) for p in inst.processors}

print(f"cheapest plan: ${base_cost:,.2f}\n")
print("bought from")
for i, v in by_mine.items():
    print(f"  {i:10} {v:6.1f} kt   {v / inst.total_demand:5.0%}")
print("refined at")
for p, v in by_proc.items():
    print(f"  {p:10} {v:6.1f} kt   {v / inst.total_demand:5.0%}   (capacity {inst.processors[p]:.0f}, price {proc_cap[p].Pi:+.2f})")

cheapest plan: $600.00

bought from
  DRC         120.0 kt    100%
  Australia     0.0 kt      0%
  Domestic      0.0 kt      0%
refined at
  China       120.0 kt    100%   (capacity 120, price +0.00)
  Domestic      0.0 kt      0%   (capacity 120, price +0.00)


Everything from one mine, through one processor. The processor is at capacity and its shadow price
is zero. Both of those are true at once; say why before going on, and what a one-kilotonne increase
in demand would do to them.

## Cap any one mine's share

Add one row per mine: its outflow may not exceed a fraction of the total requirement. The fraction
is a knob. Predict the cost at a 75% cap before running, from the second-cheapest full path.

In [8]:
SHARE_CAP = 0.75
share_rows = {i: m.addConstr(x.sum(i, "*") <= SHARE_CAP * inst.total_demand, name=f"share[{i}]") for i in inst.mines}
m.optimize()
capped_cost = m.ObjVal
print(f"cap {SHARE_CAP:.0%}: ${capped_cost:,.2f}   premium {capped_cost / base_cost - 1:.1%}")
for i in inst.mines:
    print(f"  {i:10} {sum(x[i, p].X for p in inst.processors):6.1f} kt")

cap 75%: $660.00   premium 10.0%
  DRC          90.0 kt
  Australia    30.0 kt
  Domestic      0.0 kt


## The price curve

Tighten the cap in steps and re-solve. This is one row changed per step — a loop over a knob, the
same model each time. Predict two things before running: at which cap the domestic mine, the dearest
tonne in the problem, is first bought; and at which cap the problem stops having a solution at all.

In [9]:
CAPS = [0.75, 0.60, 0.50, 0.40, 0.34, 0.33]
curve_hand = {}
print(f"{'cap':>6} {'cost':>10} {'premium':>8}   " + "  ".join(f"{i:>10}" for i in inst.mines))
for cap in CAPS:
    for i in inst.mines:
        share_rows[i].RHS = cap * inst.total_demand
    m.optimize()
    if m.Status != GRB.OPTIMAL:
        curve_hand[cap] = None
        print(f"{cap:6.0%} {'infeasible':>10}")
        continue
    curve_hand[cap] = m.ObjVal
    mix = "  ".join(f"{sum(x[i, p].X for p in inst.processors):10.1f}" for i in inst.mines)
    print(f"{cap:6.0%} {m.ObjVal:10.2f} {m.ObjVal / base_cost - 1:8.1%}   {mix}")

   cap       cost  premium          DRC   Australia    Domestic
   75%     660.00    10.0%         90.0        30.0         0.0
   60%     696.00    16.0%         72.0        48.0         0.0
   50%     720.00    20.0%         60.0        60.0         0.0
   40%     768.00    28.0%         48.0        48.0        24.0
   34%     796.80    32.8%         40.8        40.8        38.4
   33% infeasible


Three mines, each capped at the same share of the total: below one third, the three caps add to less
than the requirement and no plan exists. The table stops one step above that. Which cap would you
recommend — and what would you need to know about the DRC mine that the table cannot tell you?

## The row that was silently missing

The module's second half rebuilt this model in PyPSA and matched the cost to the cent. It also left
out the processor capacity rows: a comment said "the cap is enforced below", and nothing below
enforced it. Drop those rows here and re-solve at the base case. Predict: does the cost change?

In [10]:
for i in inst.mines:
    share_rows[i].RHS = 1.0 * inst.total_demand          # cap released
m.remove(list(proc_cap.values()))
m.optimize()
print(f"without processor capacities: ${m.ObjVal:,.2f}   (with them: ${base_cost:,.2f})")

without processor capacities: $600.00   (with them: $600.00)


Same number. Now raise the requirement at one plant by a single kilotonne and solve both versions.

In [11]:
EXTRA_KT = 1.0
plant_req["Cell Plant A"].RHS = inst.plants["Cell Plant A"] + EXTRA_KT
m.optimize()
loose = m.ObjVal
loose_china = sum(y["China", j].X for j in inst.plants)

proc_cap = {p: m.addConstr(x.sum("*", p) <= cap, name=f"processor[{p}]") for p, cap in inst.processors.items()}
m.optimize()
tight = m.ObjVal
tight_china = sum(y["China", j].X for j in inst.plants)
plant_req["Cell Plant A"].RHS = inst.plants["Cell Plant A"]        # restore the table's demand
m.optimize()

print(f"demand + {EXTRA_KT:.0f} kt, no processor rows : ${loose:,.2f}   China refines {loose_china:.1f} kt of a {inst.processors['China']:.0f} kt capacity")
print(f"demand + {EXTRA_KT:.0f} kt, with the rows    : ${tight:,.2f}   China refines {tight_china:.1f} kt")

demand + 1 kt, no processor rows : $607.00   China refines 121.0 kt of a 120 kt capacity
demand + 1 kt, with the rows    : $608.00   China refines 120.0 kt


The two models agreed on the course's data because total demand happened to equal one processor's
capacity exactly. One kilotonne later they disagree, and the one without the rows refines more than
the refinery can hold. What kind of check would have caught the missing rows on the original data?

---

# Now the streamlined version

One model, re-solved nine times with one knob changed, so the package owns it: `sourcing.solve`
takes the tables and the cap, `price_curve` runs the sweep, and `processor_caps=False` reproduces
the omission on purpose.

In [12]:
from orteach import sourcing
from orteach.tolerance import AGREEMENT_RTOL, rel_diff

pkg_base = sourcing.solve(inst, env=env)
pkg_curve = dict(sourcing.price_curve(inst, CAPS, env=env))
pkg_loose = sourcing.solve(inst, processor_caps=False, env=env)
print(f"base ${pkg_base.objective:,.2f}   by mine {pkg_base.by_mine()}")
for cap, plan in pkg_curve.items():
    print(f"  cap {cap:.0%}: " + (f"${plan.objective:,.2f}" if plan.feasible else "infeasible"))
print(f"smallest feasible equal share: {sourcing.smallest_feasible_share(inst):.4f}")

base $600.00   by mine {'DRC': 120.0, 'Australia': 0.0, 'Domestic': 0.0}
  cap 75%: $660.00
  cap 60%: $696.00
  cap 50%: $720.00
  cap 40%: $768.00
  cap 34%: $796.80
  cap 33%: infeasible
smallest feasible equal share: 0.3333


## The agreement assertion

The base cost and plan, every point on the price curve including which caps are infeasible, and the
cost without processor rows — hand-built against the package. The base optimum is unique (one path
is strictly cheapest), so the flows are compared too.

In [13]:
checks = [("base cost", base_cost, pkg_base.objective),
          ("no processor rows", m.ObjVal, pkg_loose.objective)]
for i in inst.mines:
    checks.append((f"base kt from {i}", by_mine[i], pkg_base.by_mine()[i]))
for cap in CAPS:
    if curve_hand[cap] is None:
        assert not pkg_curve[cap].feasible, f"package found a plan at cap {cap} where the hand model found none"
    else:
        checks.append((f"cap {cap:.0%}", curve_hand[cap], pkg_curve[cap].objective))

worst = max(rel_diff(h, k) for _, h, k in checks)
print(f"{len(checks)} comparisons, plus the infeasible caps")
for name, hand, packaged in checks[:4]:
    print(f"  {name:20} hand {hand:10.2f}   package {packaged:10.2f}   rel {rel_diff(hand, packaged):.2e}")
print("  ...")
assert worst < AGREEMENT_RTOL, f"notebook and package disagree by {worst:.2e}"
print(f"\nnotebook and package agree to {worst:.1e}")

10 comparisons, plus the infeasible caps
  base cost            hand     600.00   package     600.00   rel 0.00e+00
  no processor rows    hand     600.00   package     600.00   rel 0.00e+00
  base kt from DRC     hand     120.00   package     120.00   rel 0.00e+00
  base kt from Australia hand       0.00   package       0.00   rel 0.00e+00
  ...

notebook and package agree to 0.0e+00


---

## Where to take this next

- Put the share cap on the *processors* instead of the mines. Which cap is more expensive to satisfy,
  and what does that say about where the bottleneck in this industry sits?
- The domestic mine ships to China at $8.0/kt and to the domestic processor at $3.0. Find the ore
  tariff at which it enters the uncapped solution on price alone — without re-solving, from the
  reduced costs.
- The lithium supply-chain models this module grew from live in their own repository,
  `sear-labs/advopt-lithiumsc`, with stochastic demand, Benders decomposition and interdiction. Open
  its `01_deterministic` notebook and find this problem inside it.